## **Supervised Model III – XGBoost (Improved)**

Key changes from XGBoost_consolidated:
1. **Unsupervised anomaly scores** (Isolation Forest, LOF, Mahalanobis) as features → detects unseen anomaly groups
2. **SVD reconstruction error + structural features** (KL divergence, Gini, co-rating graph) → generalises across anomaly types
3. **Optuna optimised for F1 at threshold 0.5** → aligned with Codabench evaluation
4. **5-fold CV** instead of 20-fold → more stable estimates, less correlated ensemble

In [1]:
results = []

In [2]:
import matplotlib.pyplot as plt
import zipfile, warnings
import xgboost as xgb
import pandas as pd
import numpy as np
import optuna
from scipy.stats import entropy
from scipy.spatial.distance import mahalanobis
from scipy.sparse import csr_matrix
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import TruncatedSVD, NMF

warnings.filterwarnings('ignore')

TOTAL_ITEMS = 1000
RATING_RANGE = range(6)
N_SVD_COMPONENTS = 50

/Users/tori/Documents/OFFICES/SCHOOL/Y3S2/Machine_Learning_421_SMU/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### **Data Loading**

In [3]:
# ── I/O helpers ──────────────────────────────────────────────────────

def load_npz(path: str) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    data = np.load(path)
    XX = pd.DataFrame(data["X"], columns=["user", "item", "rating"])
    yy = None
    if "y" in data:
        yy = pd.DataFrame(data["y"], columns=["user", "label"])
    return XX, yy


def combine_labeled_data(
    *npz_paths: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    all_X, all_y = [], []
    for path in npz_paths:
        data = np.load(path)
        all_X.append(pd.DataFrame(data["X"], columns=["user", "item", "rating"]))
        all_y.append(pd.DataFrame(data["y"], columns=["user", "label"]))

    XX = pd.concat(all_X, ignore_index=True)
    yy = pd.concat(all_y, ignore_index=True).drop_duplicates(subset="user", keep="first")

    n_anom = int(yy["label"].sum())
    print(f"Combined {len(npz_paths)} files")
    print(f"{yy.shape[0]} users ({n_anom} anomalous, {yy.shape[0] - n_anom} normal), {XX.shape[0]} interactions")
    return XX, yy

In [4]:
# ── Sparse matrix builder ────────────────────────────────────────────

def build_user_item_matrix(XX: pd.DataFrame, user_ids: np.ndarray):
    """Build a sparse user×item rating matrix.
    
    Returns:
        mat: sparse CSR matrix (n_users × TOTAL_ITEMS)
        uid_to_row: dict mapping user_id → row index
    """
    uid_to_row = {uid: i for i, uid in enumerate(user_ids)}

    # Vectorised — avoid iterrows on 400k+ rows
    mask = XX["user"].isin(uid_to_row)
    sub = XX.loc[mask]
    rows = sub["user"].map(uid_to_row).values
    cols = sub["item"].values.astype(int)
    vals = sub["rating"].values.astype(float)

    mat = csr_matrix((vals, (rows, cols)), shape=(len(user_ids), TOTAL_ITEMS))
    return mat, uid_to_row

In [5]:
# ── Build hand-crafted features –––––––––––––––––––––––––––––────────

def compute_item_stats(XX_train: pd.DataFrame) -> dict:
    item_avg = XX_train.groupby("item")["rating"].mean().rename("item_avg_rating")
    item_pop = XX_train.groupby("item")["user"].count().rename("item_popularity")
    return {"item_avg_rating": item_avg, "item_popularity": item_pop}


def build_handcrafted_features(
    XX: pd.DataFrame,
    item_stats: dict,
    total_items: int = TOTAL_ITEMS,
) -> pd.DataFrame:
    """Original 24 features — unchanged from your notebook."""
    item_avg = item_stats["item_avg_rating"]
    item_pop = item_stats["item_popularity"]

    stats = XX.groupby("user")["rating"].agg(
        rating_mean="mean", rating_std="std", rating_median="median",
        rating_min="min", rating_max="max", rating_count="count",
    )
    stats["rating_std"] = stats["rating_std"].fillna(0)
    stats["rating_range"] = stats["rating_max"] - stats["rating_min"]

    rdist = XX.groupby(["user", "rating"]).size().unstack(fill_value=0)
    rdist = rdist.reindex(columns=RATING_RANGE, fill_value=0)
    rprops = rdist.div(rdist.sum(axis=1), axis=0)
    rprops.columns = [f"prop_rating_{i}" for i in RATING_RANGE]
    stats["rating_entropy"] = rprops.apply(
        lambda row: entropy(row.values[row.values > 0]), axis=1
    )
    stats = stats.join(rprops)
    stats["prop_extreme"] = rprops["prop_rating_0"] + rprops["prop_rating_5"]

    stats["unique_items_rated"] = XX.groupby("user")["item"].nunique()
    stats["item_coverage_ratio"] = stats["unique_items_rated"] / total_items

    XX_pop = XX.merge(item_pop, left_on="item", right_index=True, how="left")
    XX_pop["item_popularity"] = XX_pop["item_popularity"].fillna(0)
    pop_f = XX_pop.groupby("user")["item_popularity"].agg(
        avg_item_popularity="mean", std_item_popularity="std",
    )
    pop_f["std_item_popularity"] = pop_f["std_item_popularity"].fillna(0)
    stats = stats.join(pop_f)

    XX_dev = XX.merge(item_avg, left_on="item", right_index=True, how="left")
    global_train_mean = item_avg.mean()
    XX_dev["item_avg_rating"] = XX_dev["item_avg_rating"].fillna(global_train_mean)
    XX_dev["deviation"] = XX_dev["rating"] - XX_dev["item_avg_rating"]
    dev_f = XX_dev.groupby("user")["deviation"].agg(
        mean_deviation="mean", std_deviation="std",
        abs_mean_deviation=lambda x: np.mean(np.abs(x)),
    )
    dev_f["std_deviation"] = dev_f["std_deviation"].fillna(0)
    stats = stats.join(dev_f)

    iqf = XX_dev.groupby("user")["item_avg_rating"].agg(
        avg_item_avg_rating="mean", std_item_avg_rating="std",
    )
    iqf["std_item_avg_rating"] = iqf["std_item_avg_rating"].fillna(0)
    stats = stats.join(iqf)

    return stats.reset_index()

In [6]:
# ── Structural features ───────────––––––──────────────────────────────

def build_structural_features(XX: pd.DataFrame) -> pd.DataFrame:
    """Features that capture the *shape* of a user's behaviour,
    not just summary statistics.  These generalise across anomaly types."""

    feats = pd.DataFrame({"user": XX["user"].unique()})

    # ── Global rating distribution (for KL divergence) ────────────────
    global_dist = XX["rating"].value_counts(normalize=True).reindex(RATING_RANGE, fill_value=0).values
    global_dist = np.clip(global_dist, 1e-10, None)  # avoid log(0)

    def user_kl(group):
        user_dist = group["rating"].value_counts(normalize=True).reindex(RATING_RANGE, fill_value=0).values
        user_dist = np.clip(user_dist, 1e-10, None)
        return entropy(user_dist, global_dist)

    kl_df = XX.groupby("user").apply(user_kl).rename("kl_div_from_global")
    feats = feats.merge(kl_df, on="user", how="left")

    # ── Gini coefficient of item selection ────────────────────────────
    def gini_coeff(group):
        counts = group["item"].value_counts().values.astype(float)
        if len(counts) <= 1:
            return 0.0
        counts = np.sort(counts)
        n = len(counts)
        index = np.arange(1, n + 1)
        return (2.0 * np.sum(index * counts) / (n * np.sum(counts))) - (n + 1.0) / n

    gini_df = XX.groupby("user").apply(gini_coeff).rename("item_gini")
    feats = feats.merge(gini_df, on="user", how="left")

    # ── Rating "flatness" — how uniform is the rating vector? ────────
    def rating_flatness(group):
        ratings = group["rating"].values
        if len(ratings) <= 1:
            return 0.0
        # proportion of ratings equal to the mode
        vals, counts = np.unique(ratings, return_counts=True)
        return counts.max() / len(ratings)

    flat_df = XX.groupby("user").apply(rating_flatness).rename("rating_mode_frac")
    feats = feats.merge(flat_df, on="user", how="left")

    # ── Rating skewness & kurtosis ───────────────────────────────────
    skew_df = XX.groupby("user")["rating"].skew().rename("rating_skew").fillna(0)
    kurt_df = XX.groupby("user")["rating"].apply(
        lambda x: x.kurtosis() if len(x) >= 4 else 0.0
    ).rename("rating_kurtosis")
    feats = feats.merge(skew_df, on="user", how="left")
    feats = feats.merge(kurt_df, on="user", how="left")

    # ── Item overlap with popular items ──────────────────────────────
    top_items = set(XX["item"].value_counts().head(100).index)
    def top_item_frac(group):
        user_items = set(group["item"].values)
        return len(user_items & top_items) / max(len(user_items), 1)

    top_df = XX.groupby("user").apply(top_item_frac).rename("frac_top100_items")
    feats = feats.merge(top_df, on="user", how="left")

    # ── Ratings-per-item ratio (detects multi-rating anomalies) ─────
    rpi = XX.groupby("user").apply(
        lambda g: len(g) / g["item"].nunique()
    ).rename("ratings_per_item")
    feats = feats.merge(rpi, on="user", how="left")

    return feats

In [7]:
# ── SVD / NMF reconstruction error ─────────────────────–––––─────────

def build_svd_features(
    XX_ref: pd.DataFrame,
    XX_target: pd.DataFrame,
    target_users: np.ndarray,
    n_components: int = N_SVD_COMPONENTS,
) -> pd.DataFrame:
    """Fit SVD on XX_ref, compute reconstruction error for XX_target users.
    
    For fold-safe usage:
      - XX_ref    = training fold interactions (fit SVD here)
      - XX_target = ALL interactions for the users we want features for
      - target_users = the user IDs to produce features for
    """
    # ── Fit SVD on reference data ────────────────────────────────────
    ref_users = XX_ref["user"].unique()
    ref_mat, ref_uid2row = build_user_item_matrix(XX_ref, ref_users)
    
    n_comp = min(n_components, min(ref_mat.shape) - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    svd.fit(ref_mat)

    # ── Compute features for target users ────────────────────────────
    target_mat, target_uid2row = build_user_item_matrix(XX_target, target_users)
    
    # SVD reconstruction error
    latent = svd.transform(target_mat)
    reconstructed = latent @ svd.components_
    # Per-user mean squared error (only on rated items)
    target_dense = target_mat.toarray()
    mask = (target_dense != 0).astype(float)
    diff = (target_dense - reconstructed) * mask
    mse_per_user = np.sum(diff ** 2, axis=1) / np.maximum(np.sum(mask, axis=1), 1)
    mae_per_user = np.sum(np.abs(diff), axis=1) / np.maximum(np.sum(mask, axis=1), 1)
    
    # Norm of the latent representation (outliers in latent space)
    latent_norm = np.linalg.norm(latent, axis=1)
    
    # Distance from latent centroid of reference data
    ref_latent = svd.transform(ref_mat)
    centroid = ref_latent.mean(axis=0)
    latent_dist = np.linalg.norm(latent - centroid, axis=1)

    feats = pd.DataFrame({
        "user": target_users,
        "svd_recon_mse": mse_per_user,
        "svd_recon_mae": mae_per_user,
        "svd_latent_norm": latent_norm,
        "svd_latent_dist_from_centroid": latent_dist,
    })
    
    return feats, svd

In [8]:
# ── NEW: Unsupervised anomaly scores ─────────────────────────────────

def build_unsupervised_scores(
    X_ref: np.ndarray,
    X_target: np.ndarray,
) -> np.ndarray:
    """Compute unsupervised anomaly scores on X_target,
    fitted on X_ref (should be the training fold's feature matrix).
    
    Returns array of shape (n_target, 3):
      [iso_forest_score, lof_score, mahalanobis_dist]
    """
    n_target = X_target.shape[0]
    scores = np.zeros((n_target, 3))

    # ── Isolation Forest ─────────────────────────────────────────────
    iso = IsolationForest(
        n_estimators=300, contamination=0.1, random_state=42, n_jobs=-1
    )
    iso.fit(X_ref)
    # score_samples: lower = more anomalous → negate so higher = more anomalous
    scores[:, 0] = -iso.score_samples(X_target)

    # ── LOF (novelty detection mode) ─────────────────────────────────
    lof = LocalOutlierFactor(
        n_neighbors=20, contamination=0.1, novelty=True, n_jobs=-1
    )
    lof.fit(X_ref)
    scores[:, 1] = -lof.score_samples(X_target)

    # ── Mahalanobis distance ─────────────────────────────────────────
    try:
        mu = X_ref.mean(axis=0)
        cov = np.cov(X_ref, rowvar=False)
        # Regularise to avoid singular covariance
        cov += np.eye(cov.shape[0]) * 1e-6
        cov_inv = np.linalg.inv(cov)
        scores[:, 2] = np.array([
            mahalanobis(x, mu, cov_inv) for x in X_target
        ])
    except Exception:
        # Fallback: Euclidean distance from centroid
        mu = X_ref.mean(axis=0)
        scores[:, 2] = np.linalg.norm(X_target - mu, axis=1)

    return scores


UNSUP_COLS = ["iso_forest_score", "lof_score", "mahalanobis_dist"]

In [9]:
# ── Combined feature pipeline ────────────────────────────────────────

def build_all_features(
    XX: pd.DataFrame,
    item_stats: dict,
) -> pd.DataFrame:
    """Merge handcrafted + structural features for a set of users.
    SVD and unsupervised scores are added separately (they need ref/target split).
    """
    hc = build_handcrafted_features(XX, item_stats)
    st = build_structural_features(XX)
    merged = hc.merge(st, on="user", how="left")
    return merged

In [10]:
# ── Fold feature builder (used within CV to prevent data leakage) ──────

def make_fold_features(XX_raw, yy_raw, train_users, val_users):
    """Build all features for one CV fold, with proper train/val isolation."""
    XX_tr  = XX_raw[XX_raw["user"].isin(train_users)].copy()
    XX_val = XX_raw[XX_raw["user"].isin(val_users)].copy()
    yy_tr  = yy_raw[yy_raw["user"].isin(train_users)].copy()
    yy_val = yy_raw[yy_raw["user"].isin(val_users)].copy()

    # Item stats from train fold only
    item_stats_tr = compute_item_stats(XX_tr)

    # Handcrafted + structural features
    feats_tr  = build_all_features(XX_tr, item_stats_tr).merge(yy_tr, on="user")
    feats_val = build_all_features(XX_val, item_stats_tr).merge(yy_val, on="user")

    # SVD features (fit on train fold interactions)
    svd_tr, svd_model = build_svd_features(
        XX_ref=XX_tr, XX_target=XX_tr,
        target_users=feats_tr["user"].values
    )
    svd_val, _ = build_svd_features(
        XX_ref=XX_tr, XX_target=XX_val,
        target_users=feats_val["user"].values
    )
    feats_tr  = feats_tr.merge(svd_tr, on="user", how="left")
    feats_val = feats_val.merge(svd_val, on="user", how="left")

    # Determine feature columns
    feature_cols = [c for c in feats_tr.columns if c not in ["user", "label"]]
    feats_val = feats_val[["user", "label"] + feature_cols]

    # Scale
    scaler_fold = RobustScaler()
    X_tr_s  = scaler_fold.fit_transform(feats_tr[feature_cols].values)
    X_val_s = scaler_fold.transform(feats_val[feature_cols].values)

    # Unsupervised anomaly scores (fit on scaled train features)
    unsup_tr  = build_unsupervised_scores(X_tr_s, X_tr_s)
    unsup_val = build_unsupervised_scores(X_tr_s, X_val_s)

    # Append unsupervised scores as extra columns
    X_tr_final  = np.hstack([X_tr_s, unsup_tr])
    X_val_final = np.hstack([X_val_s, unsup_val])
    feature_cols_all = feature_cols + UNSUP_COLS

    y_tr  = feats_tr["label"].values
    y_val = feats_val["label"].values

    return (
        X_tr_final, y_tr, X_val_final, y_val,
        item_stats_tr, feature_cols_all, scaler_fold, svd_model
    )

In [11]:
# ── Evaluation helper ────────────────────────────────────────────────

def codabench_metrics(test_labels, scores, model_name, verbose=False):
    test_labels = np.asarray(test_labels).astype(int)
    scores = np.asarray(scores).astype(float)
    preds = (scores >= 0.5).astype(int)
    metrics = {
        "model": model_name,
        "AUC": roc_auc_score(test_labels, scores),
        "Precision": precision_score(test_labels, preds, zero_division=0),
        "Recall": recall_score(test_labels, preds, zero_division=0),
        "F1": f1_score(test_labels, preds, zero_division=0),
        "threshold": 0.5,
    }
    if verbose:
        print(f"{model_name} (Codabench t=0.5)")
        for k in ["AUC", "Precision", "Recall", "F1"]:
            print(f"# {k+':':12s} {metrics[k]:.4f}")
    return metrics

In [12]:
# Phase 3 data loading
XX_all, yy_all = combine_labeled_data(
    "data/training_batch_with_labels.npz",
    "data/first_batch_with_labels.npz",
    "data/second_batch_with_labels.npz",
)

Combined 3 files
3060 users (260 anomalous, 2800 normal), 479433 interactions


In [13]:
# Build full feature set for Optuna (all training data)
item_stats_full = compute_item_stats(XX_all)
full_train_df = build_all_features(XX_all, item_stats_full).merge(yy_all, on="user")

# SVD on full training data
all_users = full_train_df["user"].values
svd_full_df, svd_full_model = build_svd_features(
    XX_ref=XX_all, XX_target=XX_all, target_users=all_users
)
full_train_df = full_train_df.merge(svd_full_df, on="user", how="left")

feature_cols = [c for c in full_train_df.columns if c not in ["user", "label"]]

X_trainval = full_train_df[feature_cols].values
y_trainval = full_train_df["label"].values

scaler = RobustScaler()
X_trainval_s = scaler.fit_transform(X_trainval)

# Add unsupervised scores
unsup_full = build_unsupervised_scores(X_trainval_s, X_trainval_s)
X_trainval_s = np.hstack([X_trainval_s, unsup_full])
feature_cols_all = feature_cols + UNSUP_COLS

print(f"Training users: {len(y_trainval)}")
print(f"Features:       {len(feature_cols_all)}")
print(f"Feature names:  {feature_cols_all}")

Training users: 3060
Features:       38
Feature names:  ['rating_mean', 'rating_std', 'rating_median', 'rating_min', 'rating_max', 'rating_count', 'rating_range', 'rating_entropy', 'prop_rating_0', 'prop_rating_1', 'prop_rating_2', 'prop_rating_3', 'prop_rating_4', 'prop_rating_5', 'prop_extreme', 'unique_items_rated', 'item_coverage_ratio', 'avg_item_popularity', 'std_item_popularity', 'mean_deviation', 'std_deviation', 'abs_mean_deviation', 'avg_item_avg_rating', 'std_item_avg_rating', 'kl_div_from_global', 'item_gini', 'rating_mode_frac', 'rating_skew', 'rating_kurtosis', 'frac_top100_items', 'ratings_per_item', 'svd_recon_mse', 'svd_recon_mae', 'svd_latent_norm', 'svd_latent_dist_from_centroid', 'iso_forest_score', 'lof_score', 'mahalanobis_dist']


#### **Optuna Hyperparameter Search (F1-optimised)**

In [14]:
spw_global = np.sum(y_trainval == 0) / np.sum(y_trainval == 1)

def objective(trial):
    grow_policy = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])

    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 100, 4000),
        learning_rate     = trial.suggest_float("learning_rate", 0.001, 0.15, log=True),
        subsample         = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        colsample_bylevel = trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        colsample_bynode  = trial.suggest_float("colsample_bynode", 0.4, 1.0),
        min_child_weight  = trial.suggest_int("min_child_weight", 1, 10),
        gamma             = trial.suggest_float("gamma", 0.0, 2.0),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        max_delta_step    = trial.suggest_int("max_delta_step", 0, 10),
        scale_pos_weight  = trial.suggest_float("scale_pos_weight", spw_global * 0.5, spw_global * 3.0),
        grow_policy       = grow_policy,
    )

    if grow_policy == "lossguide":
        params["max_leaves"] = trial.suggest_int("max_leaves", 16, 512)
    else:
        params["max_depth"] = trial.suggest_int("max_depth", 3, 12)

    # ── 5-fold CV, optimise for F1 at t=0.5 ──────────––––––───────
    # NOTE: X_trainval_s has unsupervised scores computed on the full set,
    # so there's minor leakage here. This is a pragmatic tradeoff —
    # recomputing IF/LOF per fold per trial would be prohibitively slow.
    # The actual CV in the next section uses proper fold-isolated scores.
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=trial.number)
    fold_f1s = []
    fold_aucs = []

    for tr_i, val_i in cv.split(X_trainval_s, y_trainval):
        m = xgb.XGBClassifier(
            **params,
            eval_metric="aucpr",
            early_stopping_rounds=50,
            random_state=42,
            n_jobs=-1,
            tree_method="hist",
        )
        m.fit(
            X_trainval_s[tr_i], y_trainval[tr_i],
            eval_set=[(X_trainval_s[val_i], y_trainval[val_i])],
            verbose=False,
        )
        proba = m.predict_proba(X_trainval_s[val_i])[:, 1]
        preds = (proba >= 0.5).astype(int)
        fold_f1s.append(f1_score(y_trainval[val_i], preds, zero_division=0))
        fold_aucs.append(roc_auc_score(y_trainval[val_i], proba))

    # Primary objective: F1.  Report AUC as secondary metric.
    trial.set_user_attr("mean_auc", np.mean(fold_aucs))
    return np.mean(fold_f1s)

In [15]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=300, show_progress_bar=True)

best_params = study.best_params
print(f"Best CV F1:  {study.best_value:.4f}")
print(f"Best CV AUC: {study.best_trial.user_attrs['mean_auc']:.4f}")
print("Best params:", best_params)

[I 2026-03-29 11:15:35,188] A new study created in memory with name: no-name-48b38daf-ba6e-49af-ae82-c066ec5f0a5f
Best trial: 0. Best value: 0.102857:   0%|          | 1/300 [00:03<16:21,  3.28s/it]

[I 2026-03-29 11:15:38,471] Trial 0 finished with value: 0.10285714285714284 and parameters: {'grow_policy': 'lossguide', 'n_estimators': 1080, 'learning_rate': 0.0011235691290018592, 'subsample': 0.8304544630874152, 'colsample_bytree': 0.9483685795934123, 'colsample_bylevel': 0.9084002515245195, 'colsample_bynode': 0.7935268416374845, 'min_child_weight': 10, 'gamma': 1.4313165288248069, 'reg_alpha': 0.01144791258425006, 'reg_lambda': 1.2247671467619778, 'max_delta_step': 1, 'scale_pos_weight': 6.408628652877414, 'max_leaves': 145}. Best is trial 0 with value: 0.10285714285714284.


Best trial: 1. Best value: 0.722828:   1%|          | 2/300 [00:07<18:33,  3.74s/it]

[I 2026-03-29 11:15:42,525] Trial 1 finished with value: 0.7228275998627549 and parameters: {'grow_policy': 'depthwise', 'n_estimators': 757, 'learning_rate': 0.027606795139230807, 'subsample': 0.5757345849532312, 'colsample_bytree': 0.6027564262115875, 'colsample_bylevel': 0.4300510201676121, 'colsample_bynode': 0.42116575387126, 'min_child_weight': 7, 'gamma': 1.2054242349449342, 'reg_alpha': 0.0414395857570164, 'reg_lambda': 8.639499436865163, 'max_delta_step': 5, 'scale_pos_weight': 10.137195480892457, 'max_depth': 12}. Best is trial 1 with value: 0.7228275998627549.


Best trial: 1. Best value: 0.722828:   1%|          | 3/300 [00:11<18:20,  3.71s/it]

...

[I 2026-03-29 11:31:32,305] Trial 297 finished with value: 0.7518242274701493 and parameters: {'grow_policy': 'lossguide', 'n_estimators': 629, 'learning_rate': 0.09045439629175495, 'subsample': 0.7854780075472769, 'colsample_bytree': 0.5782281750375822, 'colsample_bylevel': 0.7431474553278182, 'colsample_bynode': 0.6422059374034005, 'min_child_weight': 1, 'gamma': 0.36028798523227956, 'reg_alpha': 0.7505301796962548, 'reg_lambda': 0.7581461297097886, 'max_delta_step': 9, 'scale_pos_weight': 13.893217249098972, 'max_leaves': 256}. Best is trial 55 with value: 0.7840744887925671.


Best trial: 298. Best value: 0.790467: 100%|█████████▉| 299/300 [16:00<00:03,  3.38s/it]

[I 2026-03-29 11:31:36,101] Trial 298 finished with value: 0.7904669496664644 and parameters: {'grow_policy': 'lossguide', 'n_estimators': 693, 'learning_rate': 0.08745827079191214, 'subsample': 0.7637591385135941, 'colsample_bytree': 0.6166260921213347, 'colsample_bylevel': 0.7535234793294785, 'colsample_bynode': 0.6208849913881677, 'min_child_weight': 1, 'gamma': 0.30357947591191914, 'reg_alpha': 0.5972134721282859, 'reg_lambda': 0.4885042309480399, 'max_delta_step': 9, 'scale_pos_weight': 14.690407955879756, 'max_leaves': 271}. Best is trial 298 with value: 0.7904669496664644.


Best trial: 298. Best value: 0.790467: 100%|██████████| 300/300 [16:04<00:00,  3.21s/it]

[I 2026-03-29 11:31:39,431] Trial 299 finished with value: 0.774348836862546 and parameters: {'grow_policy': 'lossguide', 'n_estimators': 745, 'learning_rate': 0.08873440345289003, 'subsample': 0.763403762741035, 'colsample_bytree': 0.586482871296622, 'colsample_bylevel': 0.7546806366959522, 'colsample_bynode': 0.6222050554039957, 'min_child_weight': 1, 'gamma': 0.3325014062775599, 'reg_alpha': 0.5210876252995333, 'reg_lambda': 0.620436958414082, 'max_delta_step': 9, 'scale_pos_weight': 14.79811110327535, 'max_leaves': 285}. Best is trial 298 with value: 0.7904669496664644.
Best CV F1:  0.7905
Best CV AUC: 0.9716
Best params: {'grow_policy': 'lossguide', 'n_estimators': 693, 'learning_rate': 0.08745827079191214, 'subsample': 0.7637591385135941, 'colsample_bytree': 0.6166260921213347, 'colsample_bylevel': 0.7535234793294785, 'colsample_bynode': 0.6208849913881677, 'min_child_weight': 1, 'gamma': 0.30357947591191914, 'reg_alpha': 0.5972134721282859, 'reg_lambda': 0.4885042309480399, 'max

#### **Cross-Validation (5-fold)**

In [16]:
N_FOLDS = 5
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_scores = np.zeros(len(y_trainval))
fold_aucs = []
fold_f1s  = []
fold_artifacts = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(full_train_df[feature_cols].values, y_trainval), 1):
    train_users = full_train_df.iloc[tr_idx]["user"].values
    val_users   = full_train_df.iloc[va_idx]["user"].values

    # Build fold-specific features with proper isolation
    X_tr, y_tr, X_va, y_va, item_stats_tr, feat_cols_fold, scaler_fold, svd_fold = \
        make_fold_features(XX_all, yy_all, train_users, val_users)

    model = xgb.XGBClassifier(
        **best_params,
        eval_metric="auc",
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
    )
    model.fit(X_tr, y_tr, verbose=False)

    p_va = model.predict_proba(X_va)[:, 1]
    oof_scores[va_idx] = p_va

    fold_auc = roc_auc_score(y_va, p_va)
    fold_f1  = f1_score(y_va, (p_va >= 0.5).astype(int), zero_division=0)
    fold_aucs.append(fold_auc)
    fold_f1s.append(fold_f1)

    fold_artifacts.append({
        "model": model,
        "item_stats": item_stats_tr,
        "feature_cols": feat_cols_fold,
        "scaler": scaler_fold,
        "svd": svd_fold,
    })

    print(f"Fold {fold:02d}  AUC: {fold_auc:.4f}  F1: {fold_f1:.4f}")

print(f"\nMean fold AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"Mean fold F1:  {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")
print(f"OOF AUC:       {roc_auc_score(y_trainval, oof_scores):.4f}")
print(f"OOF F1:        {f1_score(y_trainval, (oof_scores >= 0.5).astype(int), zero_division=0):.4f}")

Fold 01  AUC: 0.9401  F1: 0.7207
Fold 02  AUC: 0.9700  F1: 0.7850
Fold 03  AUC: 0.9632  F1: 0.6909
Fold 04  AUC: 0.9605  F1: 0.8037
Fold 05  AUC: 0.9640  F1: 0.7350

Mean fold AUC: 0.9596 ± 0.0102
Mean fold F1:  0.7471 ± 0.0416
OOF AUC:       0.9595
OOF F1:        0.7464


#### **Results Calibration**

In [17]:
calibrator = LogisticRegression(max_iter=1000, random_state=42)
calibrator.fit(oof_scores.reshape(-1, 1), y_trainval)

oof_scores_cal = calibrator.predict_proba(oof_scores.reshape(-1, 1))[:, 1]
print(f"Calibrated OOF AUC: {roc_auc_score(y_trainval, oof_scores_cal):.4f}")
print(f"Calibrated OOF F1:  {f1_score(y_trainval, (oof_scores_cal >= 0.5).astype(int), zero_division=0):.4f}")

Calibrated OOF AUC: 0.9595
Calibrated OOF F1:  0.7646


In [18]:
# Threshold analysis
thresholds = np.linspace(0.01, 0.99, 500)
j_scores, f1_scores_t, prec_list, rec_list = [], [], [], []

for t in thresholds:
    preds_t = (oof_scores_cal >= t).astype(int)
    tp = np.sum((preds_t == 1) & (y_trainval == 1))
    tn = np.sum((preds_t == 0) & (y_trainval == 0))
    fp = np.sum((preds_t == 1) & (y_trainval == 0))
    fn = np.sum((preds_t == 0) & (y_trainval == 1))
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    f1_t = 2 * prec * sens / (prec + sens + 1e-9)
    j_scores.append(sens + spec - 1)
    f1_scores_t.append(f1_t)
    prec_list.append(prec)
    rec_list.append(sens)

best_j_idx  = np.argmax(j_scores)
best_f1_idx = np.argmax(f1_scores_t)

print(f"Optimal threshold — Youden's J : {thresholds[best_j_idx]:.3f}  "
      f"(prec={prec_list[best_j_idx]:.3f}, rec={rec_list[best_j_idx]:.3f})")
print(f"Optimal threshold — Best F1   : {thresholds[best_f1_idx]:.3f}  "
      f"(prec={prec_list[best_f1_idx]:.3f}, rec={rec_list[best_f1_idx]:.3f})")

Optimal threshold — Youden's J : 0.030  (prec=0.466, rec=0.888)
Optimal threshold — Best F1   : 0.487  (prec=0.803, rec=0.738)


#### **Model Predictions**

In [19]:
# Load test data
XX_test, _ = load_npz("data/third_batch.npz")

In [20]:
# ── Build reference features ONCE from full training data ──────────
# (used as the reference population for unsupervised anomaly scoring)
ref_df = build_all_features(XX_all, item_stats_full)
svd_ref_df, _ = build_svd_features(
    XX_ref=XX_all, XX_target=XX_all, target_users=ref_df["user"].values
)
ref_df = ref_df.merge(svd_ref_df, on="user", how="left")

# ── Ensemble predictions across folds ─────────────────────────────
fold_test_preds = []

for fold_id, art in enumerate(fold_artifacts, 1):
    model           = art["model"]
    item_stats_fold = art["item_stats"]
    feat_cols_fold  = art["feature_cols"]
    scaler_fold     = art["scaler"]

    # Determine the base (non-unsupervised) feature columns
    base_cols = [c for c in feat_cols_fold if c not in UNSUP_COLS]

    # ── Test features (handcrafted + structural + SVD) ─────────────
    test_df = build_all_features(XX_test, item_stats_fold)
    svd_test_df, _ = build_svd_features(
        XX_ref=XX_all, XX_target=XX_test,
        target_users=test_df["user"].values,
    )
    test_df = test_df.merge(svd_test_df, on="user", how="left")

    for c in base_cols:
        if c not in test_df.columns:
            test_df[c] = 0.0

    X_test_base_s = scaler_fold.transform(test_df[base_cols].values)

    # ── Unsupervised scores (reference = full training, target = test)
    X_ref_base_s = scaler_fold.transform(
        ref_df.reindex(columns=base_cols, fill_value=0).values
    )
    unsup_test = build_unsupervised_scores(X_ref_base_s, X_test_base_s)

    X_test_final = np.hstack([X_test_base_s, unsup_test])

    p_test = model.predict_proba(X_test_final)[:, 1]
    fold_test_preds.append(p_test)
    print(f"Generated test predictions from fold {fold_id:02d}")

fold_test_preds = np.column_stack(fold_test_preds)
y_score_raw = fold_test_preds.mean(axis=1)

# Calibrate
y_score_cal = calibrator.predict_proba(y_score_raw.reshape(-1, 1))[:, 1]

# Normalise to [0,1]
y_score_norm = (y_score_cal - y_score_cal.min()) / (y_score_cal.max() - y_score_cal.min() + 1e-9)

print(f"\nTest prediction shape: {y_score_norm.shape}")
print(f"Prediction range: [{y_score_norm.min():.4f}, {y_score_norm.max():.4f}]")

Generated test predictions from fold 01
Generated test predictions from fold 02
Generated test predictions from fold 03
Generated test predictions from fold 04
Generated test predictions from fold 05

Test prediction shape: (1625,)
Prediction range: [0.0000, 1.0000]


#### **Evaluation (local/Codabench)**

In [21]:
# Save submission
np.savez("submission.npz", predictions=y_score_norm)
with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("submission.npz", arcname="submission.npz")
pd.DataFrame({"predictions": y_score_norm}).to_csv("submission.csv", index=False)
print("submission.zip ready for Codabench")

submission.zip ready for Codabench


# **XGBoost_w_unsup_features.zip**
AUC:       0.9339
Precision: 0.7465
Recall:    0.4240
F1 Score:  0.5408